### Notebook 21

# **Cross-Validation Statistical Tests**

## **1. Machine Learning**

We perform **Wilcoxon signed-rank tests** on the results of the machine learning models for the classification and regression tasks. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import scipy
import pingouin
import os

In [2]:
# Define the preprocessed data path. 
preprocessed_data_path = '../neuropolis-x1_preprocessed_data/'

# Define the results path. 
results_path = '../neuropolis-x1_results/'

In [3]:
# Load the targets. 
with open(preprocessed_data_path + 'dict_targets.p', 'rb') as file:
    dict_targets = pickle.load(file)
with open(preprocessed_data_path + 'classification/dict_targets_classification_sequence.p', 'rb') as file:
    dict_targets_class = pickle.load(file)
with open(preprocessed_data_path + 'regression/dict_targets_regression_sequence.p', 'rb') as file:
    dict_targets_reg = pickle.load(file)
with open(preprocessed_data_path + 'classification/dict_targets_classification_basis.p', 'rb') as file:
    dict_targets_foundation_models = pickle.load(file)

# Define the list of subjects, removing sub-xp102 who has a missing condition. 
subjects = ['sub-xp1' + str(x).zfill(2) for x in range(1, 11)]
subjects.remove('sub-xp102')
subject = subjects[0]

# Retrieve and display the brain region names and the number of brain regions. 
brain_regions = list(dict_targets[subject]['eegfmriNF'].keys())
print(brain_regions)
print(len(brain_regions), 'brain regions')

['Background', 'Frontal Pole', 'Insular Cortex', 'Superior Frontal Gyrus', 'Middle Frontal Gyrus', 'Inferior Frontal Gyrus, pars triangularis', 'Inferior Frontal Gyrus, pars opercularis', 'Precentral Gyrus', 'Temporal Pole', 'Superior Temporal Gyrus, anterior division', 'Superior Temporal Gyrus, posterior division', 'Middle Temporal Gyrus, anterior division', 'Middle Temporal Gyrus, posterior division', 'Middle Temporal Gyrus, temporooccipital part', 'Inferior Temporal Gyrus, anterior division', 'Inferior Temporal Gyrus, posterior division', 'Inferior Temporal Gyrus, temporooccipital part', 'Postcentral Gyrus', 'Superior Parietal Lobule', 'Supramarginal Gyrus, anterior division', 'Supramarginal Gyrus, posterior division', 'Angular Gyrus', 'Lateral Occipital Cortex, superior division', 'Lateral Occipital Cortex, inferior division', 'Intracalcarine Cortex', 'Frontal Medial Cortex', 'Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)', 'Subcallosal Cortex', 'Paracingulate

In [4]:
# Define a function to create the DataFrame for the statistical tests. 
def create_df(dict_targets, dict_predictions, brain_regions, score_type, model_type, test_set):

    # Define Pandas DataFrames to store the results. 
    df = pd.DataFrame(columns = ['subject', 'brain_region', 'model_' + score_type, 'baseline_' + score_type])
    counter = 0

    # Iterate through all subjects. 
    for subject in subjects:

        # Iterate through all brain regions. 
        for brain_region_index in range(len(brain_regions)):

            # Fill the DataFrame with the subject and brain region. 
            df.loc[counter, 'subject'] = subject
            df.loc[counter, 'brain_region'] = brain_regions[brain_region_index]

            # Retrieve the targets and predictions values for the current subject and brain region. 
            targets_values = dict_targets[subject][test_set][:, brain_region_index]
            if model_type == 'machine_learning':
                predictions_values = dict_predictions[subject][test_set][:, brain_region_index]
            elif model_type == 'deep_learning':
                predictions_values = dict_predictions[subject][:, brain_region_index]

            # For the classification task, compute the accuracy. 
            if score_type == 'accuracy':
                model_score = np.mean(targets_values == predictions_values)
                baseline_score = np.mean(targets_values == 1)
                
            # For the regression task, compute the mean absolute error (MAE).
            elif score_type == 'MAE':
                model_score = np.mean(np.abs(targets_values - predictions_values))
                baseline_score = np.mean(np.abs(targets_values - np.mean(targets_values)))

            # Fill the DataFrame with the model score and baseline score. 
            df.loc[counter, 'model_' + score_type] = model_score
            df.loc[counter, 'baseline_' + score_type] = baseline_score

            # Increment. 
            counter += 1

    # Return the DataFrame. 
    return df

In [5]:
# Define a function to perform a Wilcoxon signed-rank test. 
def wilcoxon_test(df, score_type):

    # Compute the mean and standard deviation of the model and the baseline. 
    mean_model_score = df['model_' + score_type].mean()
    mean_baseline_score = df['baseline_' + score_type].mean()
    std_model_score = df['model_' + score_type].std()
    std_baseline_score = df['baseline_' + score_type].std()

    # Perform the Wilcoxon signed-rank test using SciPy. 
    x = df['model_' + score_type].astype(float).values
    y = df['baseline_' + score_type].astype(float).values
    if score_type == 'accuracy':
        wilcoxon_stat, p_value = scipy.stats.wilcoxon(x, y, alternative = 'greater')
    elif score_type == 'MAE':
        wilcoxon_stat, p_value = scipy.stats.wilcoxon(x, y, alternative = 'less')

    # Compute the confidence interval of the mean. 
    res = scipy.stats.bootstrap((df['model_' + score_type],), np.mean, confidence_level = 0.95)
    ci_low, ci_high = res.confidence_interval

    # Store the statistics in a DataFrame. 
    statistics_scipy = pd.DataFrame([{
        'N': len(df),
        'mean_model_' + score_type: mean_model_score,
        'mean_baseline_' + score_type: mean_baseline_score,
        'std_model_' + score_type: std_model_score,
        'std_baseline_' + score_type: std_baseline_score,
        'wilcoxon_stat': wilcoxon_stat,
        'p_value': p_value,
        'ci_low': ci_low, 
        'ci_high': ci_high
    }])

    # Perform the same test using Pingouin to obtain the RBC (Rank-Biserial Correlation) and CLES (Common Language Effect Size). 
    if score_type == 'accuracy':
        statistics_pingouin = pingouin.wilcoxon(x, y, alternative = 'greater')
    elif score_type == 'MAE':
        statistics_pingouin = pingouin.wilcoxon(y, x, alternative = 'greater') # Invert x and y for MAE to get the correct effect size direction. 
    
    # Return the statistics. 
    return statistics_scipy, statistics_pingouin

In [6]:
# Define a function to create a summary DataFrame. 
def create_summary_df(model_names, score_type):
    
    # Create the summary DataFrame. 
    df_summary = pd.DataFrame(index = model_names, 
                              columns = ['N', 
                                         'mean model ' + score_type, 
                                         'mean baseline ' + score_type, 
                                         'STD model ' + score_type,
                                         'STD baseline ' + score_type,
                                         'Wilcoxon W statistic', 
                                         'p-value', 
                                         'CI (lower)',
                                         'CI (upper)', 
                                         'RBC', 
                                         'CLES'])
    
    # For regression, add a Pearson correlation column. 
    if score_type == 'MAE':
        df_summary['Pearson r'] = np.nan
    
    return df_summary

In [7]:
# Define a function to store statistics in the summary DataFrame. 
def store_statistics_in_summary_df(df_summary, model_names, model_index, statistics_scipy, statistics_pingouin, score_type):

    # Store the statistics in the summary DataFrame. 
    df_summary.loc[model_names[model_index], 'N'] = statistics_scipy['N'].values[0]
    df_summary.loc[model_names[model_index], 'mean model ' + score_type] = statistics_scipy['mean_model_' + score_type].values[0]
    df_summary.loc[model_names[model_index], 'mean baseline ' + score_type] = statistics_scipy['mean_baseline_' + score_type].values[0]
    df_summary.loc[model_names[model_index], 'STD model ' + score_type] = statistics_scipy['std_model_' + score_type].values[0]
    df_summary.loc[model_names[model_index], 'STD baseline ' + score_type] = statistics_scipy['std_baseline_' + score_type].values[0]
    df_summary.loc[model_names[model_index], 'Wilcoxon W statistic'] = statistics_scipy['wilcoxon_stat'].values[0]
    df_summary.loc[model_names[model_index], 'p-value'] = statistics_scipy['p_value'].values[0]
    df_summary.loc[model_names[model_index], 'CI (lower)'] = statistics_scipy['ci_low'].values[0]
    df_summary.loc[model_names[model_index], 'CI (upper)'] = statistics_scipy['ci_high'].values[0]
    df_summary.loc[model_names[model_index], 'RBC'] = statistics_pingouin['RBC'].values[0]
    df_summary.loc[model_names[model_index], 'CLES'] = statistics_pingouin['CLES'].values[0]

    # Ensure the p-value column is numeric, then format it in scientific notation. 
    df_summary['p-value'] = pd.to_numeric(df_summary['p-value'], errors = 'coerce')
    df_summary['p-value'] = df_summary['p-value'].apply(lambda x: f'{x:.2e}')

    # For regression, store the Pearson correlation coefficient. 
    if score_type == 'MAE':
        df_summary.loc[model_names[model_index], 'Pearson r'] = statistics_scipy['Pearson r'].values[0]

    return df_summary

In [8]:
# Define a function to perform a Wilcoxon signed-rank test on a series of models. 
def run_wilcoxon_tests(model_names, dict_targets, list_dict_predictions, brain_regions, score_type, model_type, test_set):

    # Create a summary DataFrame to store the statistics for all models. 
    df_summary = create_summary_df(model_names, score_type)

    # Create a dictionary to store the metrics DataFrames for all models. 
    dict_df_metrics = {}

    # Create a dictionary to store all targets and predictions for all models. 
    dict_all_targets_predictions = {}

    # Iterate through all models. 
    for model_index in range(len(model_names)):

        # Create the DataFrame and perform the Wilcoxon signed-rank test. 
        dict_predictions = list_dict_predictions[model_index]
        df = create_df(dict_targets, dict_predictions, brain_regions, score_type, model_type, test_set)
        statistics_scipy, statistics_pingouin = wilcoxon_test(df, score_type)

        # For regression, also compute the Pearson correlation coefficient between the targets and the predictions. 
        if score_type == 'MAE':
            all_targets = []
            all_predictions = []

            # Iterate through all subjects and brain regions to collect the targets and predictions. 
            for subject in subjects:
                for brain_region_index in range(len(brain_regions)):

                    # Retrieve the predictions and targets for the current subject and brain region. 
                    targets_values = dict_targets[subject][test_set][:, brain_region_index]
                    if model_type == 'machine_learning':
                        predictions_values = dict_predictions[subject][test_set][:, brain_region_index]
                    elif model_type == 'deep_learning':
                        predictions_values = dict_predictions[subject][:, brain_region_index]
                    all_targets.extend(targets_values)
                    all_predictions.extend(predictions_values)

            # Compute the Pearson correlation coefficient using SciPy. 
            pearson_r, _ = scipy.stats.pearsonr(all_targets, all_predictions)
            statistics_scipy['Pearson r'] = pd.Series([pearson_r])

            # Store all targets and predictions in the dictionary. 
            dict_all_targets_predictions[model_names[model_index]] = {
                'targets': all_targets,
                'predictions': all_predictions
            }

        # Store the statistics in the summary DataFrame. 
        df_summary = store_statistics_in_summary_df(df_summary, model_names, model_index, statistics_scipy, statistics_pingouin, score_type)

        # Store the metrics DataFrame in the dictionary.
        dict_df_metrics[model_names[model_index]] = df
        
    # Return the summary DataFrame and the dictionary. 
    return df_summary, dict_df_metrics, dict_all_targets_predictions

### Classification

In [9]:
# Define a function to load the machine learning results for the classification task. 
def load_classification_ml_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'classification/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the machine learning results for the classification task. 
    with open(iteration_result_path + 'dict_predictions_lr.p', 'rb') as file:
        dict_predictions_lr_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_knn.p', 'rb') as file:
        dict_predictions_knn_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_dt.p', 'rb') as file:
        dict_predictions_dt_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_rf.p', 'rb') as file:
        dict_predictions_rf_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_svm.p', 'rb') as file:
        dict_predictions_svm_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_xgb.p', 'rb') as file:
        dict_predictions_xgb_class = pickle.load(file)

    return [dict_predictions_lr_class, 
            dict_predictions_knn_class, 
            dict_predictions_dt_class, 
            dict_predictions_rf_class, 
            dict_predictions_svm_class, 
            dict_predictions_xgb_class, 
            test_set]

In [10]:
# Define a function to run the tests for machine learning classification models.
def run_classification_ml_tests(cv_iteration, model_names):

    # Load the machine learning results for the classification task.
    dict_predictions_lr_class, dict_predictions_knn_class, dict_predictions_dt_class, dict_predictions_rf_class, dict_predictions_svm_class, dict_predictions_xgb_class, test_set = load_classification_ml_results(cv_iteration)

    predictions_ml_class = [dict_predictions_lr_class, 
                            dict_predictions_knn_class, 
                            dict_predictions_dt_class, 
                            dict_predictions_rf_class, 
                            dict_predictions_svm_class, 
                            dict_predictions_xgb_class]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_ml_class, dict_df_metrics_ml_class, _ = run_wilcoxon_tests(model_names, dict_targets_class, predictions_ml_class, brain_regions, 'accuracy', 'machine_learning', test_set)
    
    return df_summary_ml_class, dict_df_metrics_ml_class

In [11]:
# Define the machine learning models for classification. 
model_names = ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest', 'Support Vector Machine', 'XGBoost']

In [12]:
# Iteration 1: run the tests for machine learning classification models. 
cv_iteration = 1
df_summary_ml_class_iteration_1, dict_df_metrics_ml_class_iteration_1 = run_classification_ml_tests(cv_iteration, model_names)
df_summary_ml_class_iteration_1

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Logistic Regression,441,0.525715,0.501181,0.041167,0.030517,67955.5,1.30e-18,0.521799,0.529514,0.487349,0.685044
K-Nearest Neighbors,441,0.509807,0.501181,0.035857,0.030517,51160.5,1.93e-03,0.506418,0.513111,0.162871,0.571855
Decision Tree,441,0.525469,0.501181,0.037464,0.030517,59619.0,2.61e-20,0.521986,0.528898,0.532327,0.69315
Random Forest,441,0.529467,0.501181,0.041609,0.030517,68874.5,1.96e-22,0.525633,0.533395,0.543354,0.707198
Support Vector Machine,441,0.532132,0.501181,0.043067,0.030517,57595.5,3.98e-26,0.528204,0.536141,0.625247,0.71616
XGBoost,441,0.524417,0.501181,0.04291,0.030517,66575.5,1.34e-16,0.520443,0.528485,0.457145,0.667204


In [13]:
# Iteration 2: run the tests for machine learning classification models. 
cv_iteration = 2
df_summary_ml_class_iteration_2, dict_df_metrics_ml_class_iteration_2 = run_classification_ml_tests(cv_iteration, model_names)
df_summary_ml_class_iteration_2

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Logistic Regression,441,0.510146,0.501905,0.040479,0.030561,52669.5,9.25e-04,0.506359,0.513933,0.174662,0.567693
K-Nearest Neighbors,441,0.502501,0.501905,0.036374,0.030561,45334.0,5.88e-01,0.499147,0.505914,-0.012396,0.502005
Decision Tree,441,0.508275,0.501905,0.041112,0.030561,47381.5,3.45e-02,0.504535,0.512331,0.103114,0.516313
Random Forest,441,0.520665,0.501905,0.044082,0.030561,60929.0,1.99e-12,0.516598,0.524791,0.391534,0.6301
Support Vector Machine,441,0.52146,0.501905,0.042267,0.030561,61339.5,6.07e-13,0.517521,0.525423,0.400909,0.639178
XGBoost,441,0.515452,0.501905,0.041534,0.030561,59249.0,1.60e-07,0.511583,0.519333,0.28474,0.598303


In [14]:
# Iteration 3: run the tests for machine learning classification models. 
cv_iteration = 3
df_summary_ml_class_iteration_3, dict_df_metrics_ml_class_iteration_3 = run_classification_ml_tests(cv_iteration, model_names)
df_summary_ml_class_iteration_3

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Logistic Regression,441,0.511595,0.501075,0.038331,0.028234,55506.0,1.11e-05,0.50817,0.515207,0.237923,0.580455
K-Nearest Neighbors,441,0.506908,0.501075,0.034792,0.028234,49172.0,5.13e-02,0.503682,0.510168,0.091498,0.533954
Decision Tree,441,0.505423,0.501075,0.035694,0.028234,41523.0,2.14e-01,0.502081,0.508801,0.045906,0.5288
Random Forest,441,0.519146,0.501075,0.038428,0.028234,62292.0,4.24e-12,0.515698,0.522863,0.38273,0.636396
Support Vector Machine,441,0.522512,0.501075,0.036723,0.028234,68747.0,4.72e-18,0.519041,0.525904,0.476906,0.673256
XGBoost,441,0.513161,0.501075,0.03818,0.028234,56768.0,1.05e-06,0.509655,0.516715,0.266069,0.592179


In [15]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_ml_class_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_ml_class_pooled = create_summary_df(model_names, score_type = 'accuracy')

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_ml_class_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_ml_class_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_ml_class_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, score_type = 'accuracy')

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_ml_class_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_ml_class_pooled = store_statistics_in_summary_df(df_summary_ml_class_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, score_type = 'accuracy')

In [16]:
# Diplay the pooled summary DataFrame. 
df_summary_ml_class_pooled

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Logistic Regression,1323,0.515818,0.501387,0.040593,0.02977,529570.0,1.51e-21,0.513664,0.51802,0.306127,0.611283
K-Nearest Neighbors,1323,0.506405,0.501387,0.03578,0.02977,436676.5,6.51e-03,0.504479,0.508334,0.080406,0.535709
Decision Tree,1323,0.513056,0.501387,0.039143,0.02977,451184.0,2.69e-13,0.511046,0.515269,0.239821,0.579187
Random Forest,1323,0.523093,0.501387,0.041656,0.02977,576495.0,1.55e-42,0.520852,0.525313,0.442175,0.657773
Support Vector Machine,1323,0.525368,0.501387,0.041035,0.02977,564019.0,1.17e-52,0.523139,0.527583,0.502198,0.675596
XGBoost,1323,0.517677,0.501387,0.04118,0.02977,548182.0,3.69e-26,0.515491,0.519894,0.339381,0.619296


### Regression

In [17]:
# Define a function to load the machine learning results for the regression task. 
def load_regression_ml_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'regression/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the machine learning results for the regression task. 
    with open(iteration_result_path + 'dict_predictions_lr.p', 'rb') as file:
        dict_predictions_lr_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_ridge.p', 'rb') as file:
        dict_predictions_ridge_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_lasso.p', 'rb') as file:
        dict_predictions_lasso_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_knn.p', 'rb') as file:
        dict_predictions_knn_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_dt.p', 'rb') as file:
        dict_predictions_dt_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_rf.p', 'rb') as file:
        dict_predictions_rf_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_svm.p', 'rb') as file:
        dict_predictions_svm_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_xgb.p', 'rb') as file:
        dict_predictions_xgb_reg = pickle.load(file)

    return [dict_predictions_lr_reg, 
            dict_predictions_ridge_reg, 
            dict_predictions_lasso_reg, 
            dict_predictions_knn_reg, 
            dict_predictions_dt_reg, 
            dict_predictions_rf_reg, 
            dict_predictions_svm_reg, 
            dict_predictions_xgb_reg, 
            test_set]

In [18]:
# Define a function to run the tests for machine learning regression models.
def run_regression_ml_tests(cv_iteration, model_names):
    
    # Load the machine learning results for the regression task.
    dict_predictions_lr_reg, dict_predictions_ridge_reg, dict_predictions_lasso_reg, dict_predictions_knn_reg, dict_predictions_dt_reg, dict_predictions_rf_reg, dict_predictions_svm_reg, dict_predictions_xgb_reg, test_set = load_regression_ml_results(cv_iteration)

    predictions_ml_reg = [dict_predictions_lr_reg,
                        dict_predictions_ridge_reg, 
                        dict_predictions_lasso_reg, 
                        dict_predictions_knn_reg, 
                        dict_predictions_dt_reg, 
                        dict_predictions_rf_reg, 
                        dict_predictions_svm_reg, 
                        dict_predictions_xgb_reg]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_ml_reg, dict_df_metrics_ml_reg, dict_all_targets_predictions_ml_reg = run_wilcoxon_tests(model_names, dict_targets_reg, predictions_ml_reg, brain_regions, 'MAE', 'machine_learning', test_set)
    
    return df_summary_ml_reg, dict_df_metrics_ml_reg, dict_all_targets_predictions_ml_reg

In [19]:
# Define the machine learning models for regression. 
model_names = ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest', 'Support Vector Machine', 'XGBoost']

In [20]:
# Iteration 1: run the tests for machine learning regression models. 
cv_iteration = 1
df_summary_ml_reg_iteration_1, dict_df_metrics_ml_reg_iteration_1, dict_all_targets_predictions_ml_reg_iteration_1 = run_regression_ml_tests(cv_iteration, model_names)
df_summary_ml_reg_iteration_1

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Linear Regression,441,0.906256,0.784689,0.135039,0.060883,85218.0,1.00e+00,0.893739,0.91903,-0.748761,0.201557,0.194516
Ridge Regression,441,0.905526,0.784689,0.13475,0.060883,85120.0,1.00e+00,0.892828,0.9178,-0.74675,0.202385,0.194810
Lasso Regression,441,0.785414,0.784689,0.060187,0.060883,61242.0,1.00e+00,0.77931,0.790649,-0.256749,0.496779,0.025510
K-Nearest Neighbors,441,0.793156,0.784689,0.071735,0.060883,58302.0,1.00e+00,0.786463,0.799841,-0.196417,0.487605,0.142218
Decision Tree,441,0.802029,0.784689,0.083235,0.060883,69845.0,1.00e+00,0.79423,0.809856,-0.433291,0.441169,0.111430
Random Forest,441,0.773626,0.784689,0.073432,0.060883,42635.0,1.14e-02,0.766853,0.780356,0.125086,0.581373,0.184105
Support Vector Machine,441,0.764478,0.784689,0.069583,0.060883,32932.0,1.82e-09,0.75779,0.770794,0.324201,0.617335,0.215243
XGBoost,441,0.824735,0.784689,0.083797,0.060883,76958.0,1.00e+00,0.8168,0.832417,-0.579257,0.322864,0.149146


In [21]:
# Iteration 2: run the tests for machine learning regression models. 
cv_iteration = 2
df_summary_ml_reg_iteration_2, dict_df_metrics_ml_reg_iteration_2, dict_all_targets_predictions_ml_reg_iteration_2 = run_regression_ml_tests(cv_iteration, model_names)
df_summary_ml_reg_iteration_2

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Linear Regression,441,0.893746,0.773745,0.134039,0.078837,89687.0,1.00e+00,0.880979,0.90602,-0.84047,0.19223,0.155291
Ridge Regression,441,0.893162,0.773745,0.133885,0.078837,89634.0,1.00e+00,0.880882,0.905544,-0.839382,0.193083,0.155439
Lasso Regression,441,0.774172,0.773745,0.078891,0.078837,56127.0,9.97e-01,0.765954,0.780927,-0.151784,0.497678,0.027752
K-Nearest Neighbors,441,0.798008,0.773745,0.100612,0.078837,72930.0,1.00e+00,0.788475,0.807169,-0.496599,0.407125,0.067308
Decision Tree,441,0.793696,0.773745,0.088167,0.078837,79410.0,1.00e+00,0.784797,0.801308,-0.629575,0.404384,0.077200
Random Forest,441,0.763202,0.773745,0.080572,0.078837,31963.0,1.91e-10,0.755091,0.770158,0.344086,0.569084,0.163857
Support Vector Machine,441,0.762951,0.773745,0.086506,0.078837,36365.0,1.94e-06,0.754252,0.77045,0.253753,0.567187,0.180330
XGBoost,441,0.818099,0.773745,0.089048,0.078837,86266.0,1.00e+00,0.809274,0.826175,-0.770267,0.297237,0.117526


In [22]:
# Iteration 3: run the tests for machine learning regression models. 
cv_iteration = 3
df_summary_ml_reg_iteration_3, dict_df_metrics_ml_reg_iteration_3, dict_all_targets_predictions_ml_reg_iteration_3 = run_regression_ml_tests(cv_iteration, model_names)
df_summary_ml_reg_iteration_3

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Linear Regression,441,0.925635,0.761724,0.113327,0.087861,94981.0,1.00e+00,0.915096,0.935849,-0.949108,0.113708,0.086321
Ridge Regression,441,0.925007,0.761724,0.113197,0.087861,94950.0,1.00e+00,0.914439,0.935516,-0.948472,0.114299,0.086455
Lasso Regression,441,0.76288,0.761724,0.087008,0.087861,68861.0,1.00e+00,0.75402,0.770408,-0.413099,0.49482,0.027640
K-Nearest Neighbors,441,0.789111,0.761724,0.08649,0.087861,79268.0,1.00e+00,0.780499,0.796782,-0.626661,0.390347,0.052121
Decision Tree,441,0.804263,0.761724,0.102025,0.087861,90386.0,1.00e+00,0.794168,0.813606,-0.854814,0.34444,0.007132
Random Forest,441,0.768863,0.761724,0.097692,0.087861,59862.0,1.00e+00,0.759337,0.777583,-0.22843,0.489127,0.080891
Support Vector Machine,441,0.763206,0.761724,0.089322,0.087861,50867.0,7.88e-01,0.754587,0.771289,-0.043843,0.503617,0.128486
XGBoost,441,0.830894,0.761724,0.096941,0.087861,92064.0,1.00e+00,0.821567,0.839683,-0.889248,0.255238,0.073470


In [23]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_ml_reg_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_ml_reg_pooled = create_summary_df(model_names, score_type = 'MAE')

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_ml_reg_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_ml_reg_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_ml_reg_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, score_type = 'MAE')

    # Pool all targets and predictions across cross-validation iterations. 
    all_targets_iteration_1 = dict_all_targets_predictions_ml_reg_iteration_1[model_name]['targets']
    all_predictions_iteration_1 = dict_all_targets_predictions_ml_reg_iteration_1[model_name]['predictions']
    all_targets_iteration_2 = dict_all_targets_predictions_ml_reg_iteration_2[model_name]['targets']
    all_predictions_iteration_2 = dict_all_targets_predictions_ml_reg_iteration_2[model_name]['predictions']
    all_targets_iteration_3 = dict_all_targets_predictions_ml_reg_iteration_3[model_name]['targets']
    all_predictions_iteration_3 = dict_all_targets_predictions_ml_reg_iteration_3[model_name]['predictions']
    all_targets_pooled = all_targets_iteration_1 + all_targets_iteration_2 + all_targets_iteration_3
    all_predictions_pooled = all_predictions_iteration_1 + all_predictions_iteration_2 + all_predictions_iteration_3

    # Compute the Pearson correlation coefficient using SciPy. 
    pearson_r, _ = scipy.stats.pearsonr(all_targets_pooled, all_predictions_pooled)
    statistics_scipy['Pearson r'] = pd.Series([pearson_r])

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_ml_reg_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_ml_reg_pooled = store_statistics_in_summary_df(df_summary_ml_reg_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, score_type = 'MAE')

In [24]:
# Diplay the pooled summary DataFrame. 
df_summary_ml_reg_pooled

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Linear Regression,1323,0.908546,0.773386,0.128436,0.077199,810128.0,1.00e+00,0.90144,0.915309,-0.849975,0.168393,0.146976
Ridge Regression,1323,0.907899,0.773386,0.128242,0.077199,809624.0,1.00e+00,0.901005,0.91473,-0.848824,0.169144,0.147170
Lasso Regression,1323,0.774155,0.773386,0.076691,0.077199,559025.0,1.00e+00,0.769969,0.778099,-0.276566,0.496458,0.026795
K-Nearest Neighbors,1323,0.793425,0.773386,0.087091,0.077199,629156.0,1.00e+00,0.788712,0.798045,-0.436715,0.427335,0.090125
Decision Tree,1323,0.799996,0.773386,0.091532,0.077199,718820.0,1.00e+00,0.794841,0.804739,-0.641468,0.39623,0.067157
Random Forest,1323,0.768564,0.773386,0.084557,0.077199,406752.0,1.25e-02,0.763852,0.772976,0.071158,0.545411,0.145297
Support Vector Machine,1323,0.763545,0.773386,0.082207,0.077199,358383.0,5.27e-09,0.759002,0.767794,0.181611,0.561936,0.175173
XGBoost,1323,0.824576,0.773386,0.090174,0.077199,766463.0,1.00e+00,0.819561,0.829229,-0.750263,0.290353,0.114148


## **2. Deep Learning**

We perform **Wilcoxon signed-rank tests** on the results of the deep learning models for the classification and regression tasks. 

### Classification

In [25]:
# Define a function to load the deep learning results for the classification task. 
def load_classification_dl_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'classification/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the deep learning results for the classification task. 
    with open(iteration_result_path + 'neural_networks/dict_predictions_neural_networks_class.p', 'rb') as file:
        dict_predictions_neural_networks_class = pickle.load(file)
    with open(iteration_result_path + 'convolutional_neural_networks/dict_predictions_convolutional_neural_networks_class.p', 'rb') as file:
        dict_predictions_convolutional_neural_networks_class = pickle.load(file)
    with open(iteration_result_path + 'recurrent_neural_networks/dict_predictions_recurrent_neural_networks_class.p', 'rb') as file:
        dict_predictions_recurrent_neural_networks_class = pickle.load(file)
    with open(iteration_result_path + 'transformers/dict_predictions_transformers_class.p', 'rb') as file:
        dict_predictions_transformers_class = pickle.load(file)

    return [dict_predictions_neural_networks_class, 
            dict_predictions_convolutional_neural_networks_class, 
            dict_predictions_recurrent_neural_networks_class, 
            dict_predictions_transformers_class, 
            test_set]

In [26]:
# Define a function to run the tests for deep learning classification models.
def run_classification_dl_tests(cv_iteration, model_names):

    # Load the deep learning results for the classification task.
    dict_predictions_neural_networks_class, dict_predictions_convolutional_neural_networks_class, dict_predictions_recurrent_neural_networks_class, dict_predictions_transformers_class, test_set = load_classification_dl_results(cv_iteration)

    predictions_dl_class = [dict_predictions_neural_networks_class, 
                            dict_predictions_convolutional_neural_networks_class, 
                            dict_predictions_recurrent_neural_networks_class, 
                            dict_predictions_transformers_class]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_dl_class, dict_df_metrics_dl_class, _ = run_wilcoxon_tests(model_names, dict_targets_class, predictions_dl_class, brain_regions, 'accuracy', 'deep_learning', test_set)
    
    return df_summary_dl_class, dict_df_metrics_dl_class

In [27]:
# Define the deep learning models for classification. 
model_names = ['Multi-Layer Perceptron', 'Convolutional Neural Network', 'Recurrent Neural Network', 'Transformer']

In [28]:
# Iteration 1: run the tests for deep learning classification models. 
cv_iteration = 1
df_summary_dl_class_iteration_1, dict_df_metrics_dl_class_iteration_1 = run_classification_dl_tests(cv_iteration, model_names)
df_summary_dl_class_iteration_1

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Multi-Layer Perceptron,441,0.536234,0.501181,0.03984,0.030517,75463.0,4.63e-33,0.532482,0.539893,0.66723,0.754714
Convolutional Neural Network,441,0.506475,0.501181,0.032264,0.030517,37382.0,6.82e-02,0.5036,0.509643,0.089298,0.535186
Recurrent Neural Network,441,0.510414,0.501181,0.036367,0.030517,47207.5,1.79e-04,0.507083,0.51394,0.207199,0.566662
Transformer,441,0.507609,0.501181,0.039072,0.030517,51684.5,1.89e-02,0.504114,0.511279,0.115513,0.537394


In [29]:
# Iteration 2: run the tests for deep learning classification models. 
cv_iteration = 2
df_summary_dl_class_iteration_2, dict_df_metrics_dl_class_iteration_2 = run_classification_dl_tests(cv_iteration, model_names)
df_summary_dl_class_iteration_2

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Multi-Layer Perceptron,441,0.521332,0.501905,0.045284,0.030561,58990.5,1.09e-11,0.517204,0.525574,0.38004,0.621125
Convolutional Neural Network,441,0.504138,0.501905,0.033252,0.030561,35594.0,4.71e-01,0.50104,0.507352,0.004402,0.511837
Recurrent Neural Network,441,0.515405,0.501905,0.03589,0.030561,51976.5,1.42e-07,0.512123,0.518795,0.296172,0.597498
Transformer,441,0.51004,0.501905,0.034658,0.030561,45632.5,1.47e-03,0.506826,0.513418,0.172846,0.550596


In [30]:
# Iteration 3: run the tests for deep learning classification models. 
cv_iteration = 3
df_summary_dl_class_iteration_3, dict_df_metrics_dl_class_iteration_3 = run_classification_dl_tests(cv_iteration, model_names)
df_summary_dl_class_iteration_3

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Multi-Layer Perceptron,441,0.515721,0.501075,0.03848,0.028234,58255.0,1.25e-09,0.512168,0.519438,0.336844,0.61216
Convolutional Neural Network,441,0.502712,0.501075,0.032275,0.028234,34643.5,5.44e-01,0.499802,0.505755,-0.006652,0.497432
Recurrent Neural Network,441,0.509082,0.501075,0.032436,0.028234,41711.5,1.09e-04,0.50609,0.512098,0.222046,0.561703
Transformer,441,0.509269,0.501075,0.03398,0.028234,40807.5,1.14e-03,0.506172,0.512502,0.182723,0.557859


In [31]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_dl_class_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_dl_class_pooled = create_summary_df(model_names, score_type = 'accuracy')

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_dl_class_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_dl_class_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_dl_class_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, score_type = 'accuracy')

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_dl_class_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_dl_class_pooled = store_statistics_in_summary_df(df_summary_dl_class_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, score_type = 'accuracy')

In [32]:
# Diplay the pooled summary DataFrame. 
df_summary_dl_class_pooled

,N,mean model accuracy,mean baseline accuracy,STD model accuracy,STD baseline accuracy,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES
Multi-Layer Perceptron,1323,0.524429,0.501387,0.042174,0.02977,579274.5,1.84e-47,0.522196,0.526747,0.469979,0.663176
Convolutional Neural Network,1323,0.504442,0.501387,0.032612,0.02977,322073.0,2.09e-01,0.502655,0.506184,0.027936,0.514691
Recurrent Neural Network,1323,0.511634,0.501387,0.035021,0.02977,421018.5,4.34e-13,0.509729,0.513528,0.241888,0.575424
Transformer,1323,0.508973,0.501387,0.035961,0.02977,412437.0,1.91e-06,0.50704,0.510901,0.1543,0.54854


### Regression

In [33]:
# Define a function to load the deep learning results for the regression task. 
def load_regression_dl_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'regression/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the deep learning results for the regression task. 
    with open(iteration_result_path + 'neural_networks/dict_predictions_neural_networks_reg.p', 'rb') as file:
        dict_predictions_neural_networks_reg = pickle.load(file)
    with open(iteration_result_path + 'convolutional_neural_networks/dict_predictions_convolutional_neural_networks_reg.p', 'rb') as file:
        dict_predictions_convolutional_neural_networks_reg = pickle.load(file)
    with open(iteration_result_path + 'recurrent_neural_networks/dict_predictions_recurrent_neural_networks_reg.p', 'rb') as file:
        dict_predictions_recurrent_neural_networks_reg = pickle.load(file)
    with open(iteration_result_path + 'transformers/dict_predictions_transformers_reg.p', 'rb') as file:
        dict_predictions_transformers_reg = pickle.load(file)

    return [dict_predictions_neural_networks_reg, 
            dict_predictions_convolutional_neural_networks_reg, 
            dict_predictions_recurrent_neural_networks_reg, 
            dict_predictions_transformers_reg, 
            test_set]

In [34]:
# Define a function to run the tests for deep learning regression models.
def run_regression_dl_tests(cv_iteration, model_names):

    # Load the deep learning results for the regression task.
    dict_predictions_neural_networks_reg, dict_predictions_convolutional_neural_networks_reg, dict_predictions_recurrent_neural_networks_reg, dict_predictions_transformers_reg, test_set = load_regression_dl_results(cv_iteration)

    predictions_dl_reg = [dict_predictions_neural_networks_reg, 
                            dict_predictions_convolutional_neural_networks_reg, 
                            dict_predictions_recurrent_neural_networks_reg, 
                            dict_predictions_transformers_reg]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_dl_reg, dict_df_metrics_dl_reg, dict_all_targets_predictions_dl_reg = run_wilcoxon_tests(model_names, dict_targets_reg, predictions_dl_reg, brain_regions, 'MAE', 'deep_learning', test_set)
    
    return df_summary_dl_reg, dict_df_metrics_dl_reg, dict_all_targets_predictions_dl_reg

In [35]:
# Define the deep learning models for regression. 
model_names = ['Multi-Layer Perceptron', 'Convolutional Neural Network', 'Recurrent Neural Network', 'Transformer']

In [36]:
# Iteration 1: run the tests for deep learning regression models. 
cv_iteration = 1
df_summary_dl_reg_iteration_1, dict_df_metrics_dl_reg_iteration_1, dict_all_targets_predictions_dl_reg_iteration_1 = run_regression_dl_tests(cv_iteration, model_names)
df_summary_dl_reg_iteration_1

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Multi-Layer Perceptron,441,0.788392,0.784689,0.061593,0.060883,57516.0,9.99e-01,0.782475,0.793815,-0.180287,0.490254,0.050590
Convolutional Neural Network,441,0.784914,0.784689,0.059884,0.060883,48405.0,4.52e-01,0.778799,0.790116,0.00668,0.500892,0.039132
Recurrent Neural Network,441,0.785827,0.784689,0.060248,0.060883,64395.0,1.00e+00,0.779946,0.791125,-0.321452,0.494228,0.014076
Transformer,441,0.783525,0.784689,0.0662,0.060883,47878.0,3.75e-01,0.777054,0.789439,0.017494,0.525398,0.128937


In [37]:
# Iteration 2: run the tests for deep learning regression models. 
cv_iteration = 2
df_summary_dl_reg_iteration_2, dict_df_metrics_dl_reg_iteration_2, dict_all_targets_predictions_dl_reg_iteration_2 = run_regression_dl_tests(cv_iteration, model_names)
df_summary_dl_reg_iteration_2

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Multi-Layer Perceptron,441,0.780656,0.773745,0.082527,0.078837,66551.0,1.00e+00,0.772285,0.787845,-0.365695,0.461598,0.011314
Convolutional Neural Network,441,0.7742,0.773745,0.077725,0.078837,52314.0,9.10e-01,0.766057,0.780751,-0.073537,0.499375,0.039305
Recurrent Neural Network,441,0.775154,0.773745,0.077401,0.078837,65260.0,1.00e+00,0.766992,0.781657,-0.339202,0.493981,0.011889
Transformer,441,0.772496,0.773745,0.08053,0.078837,49093.0,5.54e-01,0.764196,0.779309,-0.007439,0.522025,0.084355


In [38]:
# Iteration 3: run the tests for deep learning regression models. 
cv_iteration = 3
df_summary_dl_reg_iteration_3, dict_df_metrics_dl_reg_iteration_3, dict_all_targets_predictions_dl_reg_iteration_3 = run_regression_dl_tests(cv_iteration, model_names)
df_summary_dl_reg_iteration_3

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Multi-Layer Perceptron,441,0.766577,0.761724,0.088564,0.087861,63027.0,1.00e+00,0.757739,0.774482,-0.293379,0.478597,0.042013
Convolutional Neural Network,441,0.762596,0.761724,0.087154,0.087861,53552.0,9.64e-01,0.754023,0.770211,-0.098942,0.495982,0.051984
Recurrent Neural Network,441,0.764032,0.761724,0.0859,0.087861,75266.0,1.00e+00,0.755444,0.771413,-0.544536,0.492511,0.010285
Transformer,441,0.771922,0.761724,0.08597,0.087861,66856.0,1.00e+00,0.763343,0.779396,-0.371954,0.460338,0.063544


In [39]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_dl_reg_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_dl_reg_pooled = create_summary_df(model_names, score_type = 'MAE')

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_dl_reg_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_dl_reg_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_dl_reg_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, score_type = 'MAE')

    # Pool all targets and predictions across cross-validation iterations. 
    all_targets_iteration_1 = dict_all_targets_predictions_dl_reg_iteration_1[model_name]['targets']
    all_predictions_iteration_1 = dict_all_targets_predictions_dl_reg_iteration_1[model_name]['predictions']
    all_targets_iteration_2 = dict_all_targets_predictions_dl_reg_iteration_2[model_name]['targets']
    all_predictions_iteration_2 = dict_all_targets_predictions_dl_reg_iteration_2[model_name]['predictions']
    all_targets_iteration_3 = dict_all_targets_predictions_dl_reg_iteration_3[model_name]['targets']
    all_predictions_iteration_3 = dict_all_targets_predictions_dl_reg_iteration_3[model_name]['predictions']
    all_targets_pooled = all_targets_iteration_1 + all_targets_iteration_2 + all_targets_iteration_3
    all_predictions_pooled = all_predictions_iteration_1 + all_predictions_iteration_2 + all_predictions_iteration_3

    # Compute the Pearson correlation coefficient using SciPy. 
    pearson_r, _ = scipy.stats.pearsonr(all_targets_pooled, all_predictions_pooled)
    statistics_scipy['Pearson r'] = pd.Series([pearson_r])

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_dl_reg_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_dl_reg_pooled = store_statistics_in_summary_df(df_summary_dl_reg_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, score_type = 'MAE')

In [40]:
# Diplay the pooled summary DataFrame. 
df_summary_dl_reg_pooled

,N,mean model MAE,mean baseline MAE,STD model MAE,STD baseline MAE,Wilcoxon W statistic,p-value,CI (lower),CI (upper),RBC,CLES,Pearson r
Multi-Layer Perceptron,1323,0.778542,0.773386,0.078877,0.077199,559604.0,1.00e+00,0.773937,0.782538,-0.277889,0.476866,0.035078
Convolutional Neural Network,1323,0.773904,0.773386,0.076259,0.077199,461636.0,9.56e-01,0.76961,0.777808,-0.054173,0.498649,0.043222
Recurrent Neural Network,1323,0.775004,0.773386,0.075744,0.077199,615624.0,1.00e+00,0.770633,0.7789,-0.405813,0.493737,0.012123
Transformer,1323,0.775981,0.773386,0.078137,0.077199,490112.0,1.00e+00,0.771588,0.779927,-0.119199,0.501916,0.093652


## **3. Model Comparison**

We perform **Wilcoxon signed-rank tests** to compare the different models. 

In [41]:
# Combine the accuracy values from machine learning and deep learning models into a single dictionary. 
dict_df_metrics_class_pooled = {**dict_df_metrics_ml_class_pooled, **dict_df_metrics_dl_class_pooled}

# Combine the MAE values from machine learning and deep learning models into a single dictionary. 
dict_df_metrics_reg_pooled = {**dict_df_metrics_ml_reg_pooled, **dict_df_metrics_dl_reg_pooled}

# Remove the 'Ridge Regression' and 'Lasso Regression' models from the regression metrics dictionary. 
dict_df_metrics_reg_pooled.pop('Ridge Regression', None)
dict_df_metrics_reg_pooled.pop('Lasso Regression', None);

In [42]:
# Define the model names and the p-values matrix for classification. 
model_names = list(dict_df_metrics_class_pooled.keys())
p_value_matrix_class = pd.DataFrame(index = model_names, columns = model_names)

# Perform Wilcoxon signed-rank tests between each pair of models and retrieve the p-values matrix. 
for model_name_1 in model_names:
    for model_name_2 in model_names:
        scores_1 = dict_df_metrics_class_pooled[model_name_1]['model_accuracy'].astype(float).values
        scores_2 = dict_df_metrics_class_pooled[model_name_2]['model_accuracy'].astype(float).values
        if model_name_1 == model_name_2:
            p_value_matrix_class.loc[model_name_1, model_name_2] = '–'
        else:
            wilcoxon_stat, p_value = scipy.stats.wilcoxon(scores_1, scores_2, alternative = 'greater')
            p_value_matrix_class.loc[model_name_1, model_name_2] = f'{p_value:.2e}'

# Replace the model names with their abbreviations in the p-values matrix, and display the matrix. 
model_abbreviations = ['Logistic', 'KNN', 'DT', 'RF', 'SVM', 'XGB', 'MLP', 'CNN', 'RNN', 'Transformer']
p_value_matrix_class.index = model_abbreviations
p_value_matrix_class.columns = model_abbreviations
p_value_matrix_class

,Logistic,KNN,DT,RF,SVM,XGB,MLP,CNN,RNN,Transformer
Logistic,–,1.08e-11,3.30e-02,1.00e+00,1.00e+00,8.79e-01,1.00e+00,1.21e-15,1.55e-03,6.17e-07
KNN,1.00e+00,–,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00,2.24e-02,1.00e+00,9.67e-01
DT,9.67e-01,4.58e-06,–,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.53e-10,1.13e-01,6.59e-04
RF,2.12e-08,4.44e-31,1.16e-14,–,9.67e-01,1.22e-05,8.19e-01,1.39e-38,5.50e-18,6.55e-24
SVM,1.60e-17,5.38e-41,8.81e-21,3.30e-02,–,2.84e-12,1.44e-01,8.19e-49,3.21e-26,2.50e-31
XGB,1.21e-01,2.30e-15,9.76e-05,1.00e+00,1.00e+00,–,1.00e+00,1.37e-19,5.94e-06,5.05e-10
MLP,9.42e-13,5.11e-33,6.87e-16,1.81e-01,8.56e-01,1.04e-07,–,1.65e-38,1.83e-20,1.45e-26
CNN,1.00e+00,9.78e-01,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00,–,1.00e+00,1.00e+00
RNN,9.98e-01,2.83e-04,8.87e-01,1.00e+00,1.00e+00,1.00e+00,1.00e+00,7.18e-10,–,1.83e-02
Transformer,1.00e+00,3.33e-02,9.99e-01,1.00e+00,1.00e+00,1.00e+00,1.00e+00,3.35e-04,9.82e-01,–


In [43]:
# Define the model names and the p-values matrix for regression. 
model_names = list(dict_df_metrics_reg_pooled.keys())
p_value_matrix_reg = pd.DataFrame(index = model_names, columns = model_names)

# Perform Wilcoxon signed-rank tests between each pair of models and retrieve the p-values matrix. 
for model_name_1 in model_names:
    for model_name_2 in model_names:
        scores_1 = dict_df_metrics_reg_pooled[model_name_1]['model_MAE'].astype(float).values
        scores_2 = dict_df_metrics_reg_pooled[model_name_2]['model_MAE'].astype(float).values
        if model_name_1 == model_name_2:
            p_value_matrix_reg.loc[model_name_1, model_name_2] = '–'
        else:
            wilcoxon_stat, p_value = scipy.stats.wilcoxon(scores_1, scores_2, alternative = 'less')
            p_value_matrix_reg.loc[model_name_1, model_name_2] = f'{p_value:.2e}'

# Replace the model names with their abbreviations in the p-values matrix, and display the matrix. 
model_abbreviations = ['Linear', 'KNN', 'DT', 'RF', 'SVM', 'XGB', 'MLP', 'CNN', 'RNN', 'Transformer']
p_value_matrix_reg.index = model_abbreviations
p_value_matrix_reg.columns = model_abbreviations
p_value_matrix_reg

,Linear,KNN,DT,RF,SVM,XGB,MLP,CNN,RNN,Transformer
Linear,–,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00
KNN,2.25e-149,–,1.51e-05,1.00e+00,1.00e+00,2.65e-69,1.00e+00,1.00e+00,1.00e+00,1.00e+00
DT,7.36e-122,1.00e+00,–,1.00e+00,1.00e+00,4.21e-49,1.00e+00,1.00e+00,1.00e+00,1.00e+00
RF,1.80e-175,3.23e-53,6.90e-117,–,1.00e+00,6.50e-193,1.81e-11,1.38e-03,3.63e-05,4.80e-10
SVM,3.40e-199,2.99e-110,2.18e-76,7.72e-05,–,2.32e-164,1.06e-24,1.38e-10,3.29e-13,8.90e-21
XGB,3.58e-113,1.00e+00,1.00e+00,1.00e+00,1.00e+00,–,1.00e+00,1.00e+00,1.00e+00,1.00e+00
MLP,1.83e-153,2.63e-22,2.08e-49,1.00e+00,1.00e+00,1.93e-111,–,1.00e+00,1.00e+00,9.07e-01
CNN,4.73e-158,2.27e-39,1.82e-84,9.99e-01,1.00e+00,1.26e-123,7.19e-13,–,3.22e-14,9.63e-04
RNN,3.40e-157,8.63e-36,1.77e-77,1.00e+00,1.00e+00,2.23e-120,1.87e-08,1.00e+00,–,4.52e-02
Transformer,5.03e-175,1.03e-24,1.59e-43,1.00e+00,1.00e+00,1.17e-118,9.29e-02,9.99e-01,9.55e-01,–


## **4. Results**

In [44]:
# If the cross-validation results path does not exist, create it. 
if not os.path.exists(results_path + 'cross_validation/'):
    os.makedirs(results_path + 'cross_validation/')
    
# Store the general results in a dictionary. 
general_results = dict()

for cv_iteration in [1, 2, 3]:
    general_results['df_summary_ml_class_iteration_' + str(cv_iteration)] = globals()['df_summary_ml_class_iteration_' + str(cv_iteration)]
    general_results['df_summary_ml_reg_iteration_' + str(cv_iteration)] = globals()['df_summary_ml_reg_iteration_' + str(cv_iteration)]
    general_results['df_summary_dl_class_iteration_' + str(cv_iteration)] = globals()['df_summary_dl_class_iteration_' + str(cv_iteration)]
    general_results['df_summary_dl_reg_iteration_' + str(cv_iteration)] = globals()['df_summary_dl_reg_iteration_' + str(cv_iteration)]
general_results['df_summary_ml_class_pooled'] = df_summary_ml_class_pooled
general_results['df_summary_ml_reg_pooled'] = df_summary_ml_reg_pooled
general_results['df_summary_dl_class_pooled'] = df_summary_dl_class_pooled
general_results['df_summary_dl_reg_pooled'] = df_summary_dl_reg_pooled
general_results['p_value_matrix_class'] = p_value_matrix_class
general_results['p_value_matrix_reg'] = p_value_matrix_reg

# Save the general results into a Pickle file. 
with open(results_path + 'cross_validation/general_results.p', 'wb') as file:
    pickle.dump(general_results, file)

In [45]:
# Display the keys of the general results dictionary. 
list(general_results.keys())

['df_summary_ml_class_iteration_1',
 'df_summary_ml_reg_iteration_1',
 'df_summary_dl_class_iteration_1',
 'df_summary_dl_reg_iteration_1',
 'df_summary_ml_class_iteration_2',
 'df_summary_ml_reg_iteration_2',
 'df_summary_dl_class_iteration_2',
 'df_summary_dl_reg_iteration_2',
 'df_summary_ml_class_iteration_3',
 'df_summary_ml_reg_iteration_3',
 'df_summary_dl_class_iteration_3',
 'df_summary_dl_reg_iteration_3',
 'df_summary_ml_class_pooled',
 'df_summary_ml_reg_pooled',
 'df_summary_dl_class_pooled',
 'df_summary_dl_reg_pooled',
 'p_value_matrix_class',
 'p_value_matrix_reg']